# Evidence Recall Targeted Sweep Summary

이 노트북은 기존 eval 결과를 변경하지 않고, 별도 참고 지표로 진행한 evidence recall 실험을 정리한 문서다.

핵심 질문은 다음과 같다.

- 정답 문서를 찾았는데 정답 근거 chunk를 놓치는 문제가 얼마나 줄었는가?
- 질문 유형별로 필요한 `fact_type`을 먼저 고르는 방식이 도움이 되는가?
- 다중 문서 질문에서 한 문서가 top-k를 독점하지 않도록 조절하면 근거 recall이 좋아지는가?

이번 실험은 generation을 새로 돌린 것이 아니라, generation 전에 context 안에 정답 evidence가 들어갈 가능성을 확인한 진단 실험이다.

## 사용한 입력 파일

| 역할 | 경로 |
|---|---|
| 기존 generation/retrieval prediction | `outputs/generation/context_mode_compare_phase34_gold_qwen/rfp_target_evidence_source_store_qwen3_8b_4bit_run1_postprocessed_eval_predictions.jsonl` |
| gold evidence label | `eval/evaluation/data/rfp_domain_gold_sample.jsonl` |
| chunk sidecar | `indexes/chroma_kure_v1_chunks_v2_125/chunks.json` |
| source store | `data/processed/source_store_v2_125.jsonl` |
| 최종 요약 | `outputs/evidence_recall/targeted_evidence_sweep_final_summary.md` |

기존 `eval/` 폴더의 평가 로직은 수정하지 않았다.

## 실험에 사용한 기법

### 1. 문서 내부 evidence 재검색

기존 retrieval이 가져온 문서 후보를 유지한 뒤, 그 문서 안의 `fact_candidates` chunk를 다시 점수화했다.  
이 방식은 “정답 문서는 맞는데 chunk가 엉뚱한 경우”를 줄이기 위한 것이다.

### 2. 질문 유형별 fact_type 우선순위

질문을 keyword rule로 분류하고, 유형에 맞는 fact_type을 먼저 확보했다.

| 질문 유형 | 우선 fact_type 예시 |
|---|---|
| budget | `project_budget`, `budget`, `estimated_price`, `base_amount` |
| date | `duration`, `project_duration`, `bid_deadline`, `maintenance_period`, `warranty_period` |
| submission | `submission_documents`, `submission_logistics` |
| eligibility | `eligibility`, `qualification` |
| summary/general | `document_summary`, `business_type`, `requirements` |

### 3. 다중 문서 quota 조절

다중 문서 질문에서 한 문서의 chunk가 top-k를 독점하지 않도록, 문서별 배치 정책을 비교했다.

- `adaptive`: 문서 수에 따라 3:2, 2:2:1처럼 배분
- `doc_min1_then_score`: 문서별 최소 1개 확보 후 전체 점수순
- `doc_min2_then_rr`: 문서별 최소 2개 확보 후 round-robin
- `round_robin`: 문서별로 돌아가며 선택
- `sequential`: 첫 문서부터 가능한 만큼 선택

### 4. source_store final_* 보정

`source_store`에 있는 `final_budget`, `final_project_duration`, `final_bid_deadline` 같은 정리된 필드를 참고해 관련 fact candidate에 보정을 주었다.

## 전체 결과 요약

| 지표 | 기존 baseline | 최종 추천 조합 |
|---|---:|---:|
| evidence_hit@5 | 0.6600 | 0.8800 |
| evidence_recall@5 | 0.2096 | 0.3821 |
| evidence_recall@10 | 0.2146 | 0.5696 |
| evidence_recall@20 | 0.2146 | 0.7550 |
| context_evidence_recall | 0.2146 | 0.8875 |
| doc_hit_but_evidence_missed | 17 | 3 |

최종 추천 조합:

`existing_strict_pack_required_first_adaptive_d5_p15_k30`

해석하면, 기존에는 정답 문서를 찾고도 정답 근거 chunk가 context에 들어가지 않은 문제가 17건 있었는데, 최종 조합에서는 3건으로 줄었다.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()

p15_dir = ROOT / 'outputs/evidence_recall/targeted_evidence_sweep_p15_variation/rfp_target_evidence_source_store_qwen3_8b_4bit_run1_postprocessed_eval_predictions'
comparison_path = p15_dir / 'comparison_by_recall.csv'

df = pd.read_csv(comparison_path)
cols = [
    'variant',
    'evidence_hit_at_5',
    'evidence_recall_at_5',
    'evidence_recall_at_10',
    'evidence_recall_at_20',
    'context_evidence_recall',
    'doc_recall_at_5',
    'doc_hit_but_evidence_missed',
]
df[cols].head(10)

,variant,evidence_hit_at_5,evidence_recall_at_5,evidence_recall_at_10,evidence_recall_at_20,context_evidence_recall,doc_recall_at_5,doc_hit_but_evidence_missed
0,existing_strict_pack_required_first_adaptive_d...,0.88,0.382143,0.569643,0.755000,0.887500,0.743333,3
1,existing_strict_pack_required_first_doc_min2_t...,0.88,0.382143,0.569643,0.755000,0.887500,0.743333,3
2,existing_canonical_required_first_adaptive_d5_...,0.88,0.378810,0.559643,0.742500,0.875000,0.743333,3
3,existing_canonical_required_first_doc_min2_the...,0.88,0.378810,0.559643,0.742500,0.875000,0.743333,3
4,baseline_existing_context,0.66,0.209643,0.214643,0.214643,0.214643,0.803333,17


In [2]:
from IPython.display import HTML, display

plot_df = df[cols].head(5).copy()
metrics = ['evidence_hit_at_5', 'evidence_recall_at_5', 'context_evidence_recall']

rows = []
for _, row in plot_df.iterrows():
    rows.append(f"<h4 style='margin:16px 0 6px'>{row['variant']}</h4>")
    for metric in metrics:
        value = float(row[metric])
        width = max(2, int(value * 100))
        rows.append(
            "<div style='display:flex;align-items:center;margin:4px 0'>"
            f"<div style='width:210px;font-family:monospace'>{metric}</div>"
            f"<div style='background:#3b82f6;height:16px;width:{width * 3}px;border-radius:3px'></div>"
            f"<div style='margin-left:8px'>{value:.4f}</div>"
            "</div>"
        )

display(HTML(''.join(rows)))

## p10 / p12 / p15 비교

p10, p12, p15는 top-5 기준 `evidence_recall@5`와 `evidence_hit@5`가 같았다.  
다만 generation context 전체에 들어가는 evidence 기준으로는 p15가 가장 좋았다.

| max_per_doc | evidence_hit@5 | evidence_recall@5 | context_evidence_recall |
|---:|---:|---:|---:|
| 10 | 0.8800 | 0.3821 | 0.8575 |
| 12 | 0.8800 | 0.3821 | 0.8850 |
| 15 | 0.8800 | 0.3821 | 0.8875 |

그래서 최종 후보는 p15로 잡았다. top-5 점수는 유지하면서, 생성 모델이 참고할 수 있는 전체 context 안의 정답 근거 포함률이 조금 더 높기 때문이다.

## 왜 이 조합이 좋은가

`existing_strict_pack_required_first_adaptive_d5_p15_k30`는 다음 성격을 가진다.

- `existing`: 이미 retrieval이 잡은 문서 후보 안에서만 evidence를 다시 고른다.
- `strict_pack`: 질문 유형에 맞는 핵심 fact candidate를 강하게 우선한다.
- `required_first`: 예산 질문이면 예산 fact, 기간 질문이면 기간 fact처럼 필요한 fact_type을 먼저 확보한다.
- `adaptive`: 문서 수에 따라 top-5 slot을 적절히 배분한다.
- `d5`: 후보 문서를 최대 5개 사용한다.
- `p15`: 문서별 evidence 후보를 최대 15개까지 본다.
- `k30`: generation context 후보는 최대 30개까지 구성한다.

즉, 문서 검색 자체를 복잡하게 더 넓히기보다, 이미 잡힌 문서 안에서 정답 근거를 더 잘 고르는 방향이다.

In [3]:
best_variant = 'existing_strict_pack_required_first_adaptive_d5_p15_k30'
best_path = p15_dir / 'top_variant_predictions' / f'{best_variant}.jsonl'
best_diag_path = p15_dir / 'top_variant_predictions' / f'{best_variant}_evidence_recall_results.csv'

best_diag = pd.read_csv(best_diag_path)
failure_cols = [
    'id',
    'question',
    'diagnosis',
    'evidence_recall_at_5',
    'doc_recall_at_5',
    'missing_gold_chunk_ids_at_5',
    'retrieved_chunk_ids_at_5',
]

best_diag[best_diag['evidence_recall_at_5'] < 1][failure_cols].head(10)

,id,question,diagnosis,evidence_recall_at_5,doc_recall_at_5,missing_gold_chunk_ids_at_5,retrieved_chunk_ids_at_5
0,Q003,코이카(KOICA) 전자조달의 '우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 ...,evidence_partially_retrieved,0.375,1.000000,"[""doc_afba17272e87_fact_candidates_fact_0004_p...","[""doc_afba17272e87_fact_candidates_fact_0002_p..."
1,Q009,그랜드코리아레저(주)의 그룹웨어 시스템 구축 사업과 인천광역시의 도시계획위원회 통합...,evidence_partially_retrieved,0.375,0.500000,"[""doc_821a622dbf13_fact_candidates_fact_0003_p...","[""doc_821a622dbf13_fact_candidates_fact_0002_p..."
2,Q010,국립중앙의료원의 차세대 응급의료 상황관리시스템 사업과 한국원자력연구원의 선량평가시스...,evidence_partially_retrieved,0.125,1.000000,"[""doc_927ecfb304fd_fact_candidates_fact_0001_p...","[""doc_106b18e84aa2_fact_candidates_fact_0002_p..."
3,Q027,한국수자원공사의 '용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사'와 코이카...,evidence_partially_retrieved,0.250,1.000000,"[""doc_6a4eb959610e_fact_candidates_fact_0003_p...","[""doc_6a4eb959610e_fact_candidates_fact_0002_p..."
4,Q028,고려대학교의 '차세대 포털·학사 정보시스템 구축사업'과 인천광역시의 '도시계획위원회...,evidence_partially_retrieved,0.375,1.000000,"[""doc_821a622dbf13_fact_candidates_fact_0004_p...","[""doc_821a622dbf13_fact_candidates_fact_0002_p..."
5,Q029,"그랜드코리아레저(주)의 그룹웨어 시스템 구축 사업 예산(1,515,000,000원)...",evidence_partially_retrieved,0.250,0.500000,"[""doc_59211e6008a9_fact_candidates_fact_0001_p...","[""doc_fcb9cb7e2d84_fact_candidates_fact_0002_p..."
6,Q046,국립중앙의료원 사업에서 '차세대 응급의료 상황관리 콜센터' 구축에 9억 원이 소요되...,evidence_partially_retrieved,0.375,1.000000,"[""doc_a43e5952d388_fact_candidates_fact_0004_p...","[""doc_a43e5952d388_fact_candidates_fact_0002_p..."
7,Q050,그랜드코리아레저(주)의 그룹웨어 시스템 구축 예산과 한국원자력연구원의 선량평가시스템...,evidence_partially_retrieved,0.250,0.500000,"[""doc_927ecfb304fd_fact_candidates_fact_0003_p...","[""doc_fcb9cb7e2d84_fact_candidates_fact_0002_p..."
8,Q053,"고려대학교(112억), 코이카(67억), 아시아물위원회(50억)의 3개 사업은 모두...",doc_hit_but_evidence_missed,0.000,0.333333,"[""doc_5b75765779be_fact_candidates_fact_0001_p...","[""doc_c6761a6fb2b7_fact_candidates_fact_0002_p..."
9,Q067,고려대학교의 '차세대 포털·학사 정보시스템 구축사업'과 부산국제영화제의 '온라인서비...,evidence_partially_retrieved,0.500,0.500000,"[""doc_c6761a6fb2b7_fact_candidates_fact_0003_p...","[""doc_59211e6008a9_fact_candidates_fact_0002_p..."


## 다음 단계

이 결과는 retrieval/evidence 진단 기준으로는 의미 있게 좋아졌다.  
하지만 최종 목표는 generation 답변 품질이므로, 다음 단계는 이 variant를 generation context에 연결해 실제 답변을 비교하는 것이다.

추천 비교군:

1. 기존 `rfp_recommended` context
2. 기존 `rfp_recommended + source_store`
3. 이번 최종 evidence variant: `existing_strict_pack_required_first_adaptive_d5_p15_k30`

20개 고정 샘플로 먼저 비교하고, 좋아지는 유형과 나빠지는 유형을 확인한 뒤 50개 전체로 넓히는 순서가 안전하다.